# 01 Create IAQ Dataset

Generates the synthetic indoor air quality dataset using literature-backed emission factors.

**Run order:** this notebook is part of the AirAware workflow and uses the shared repository folders: `data/`, `data/processed/`, and `outputs/`.

# Synthetic Indoor Air Quality Dataset Generator

This Dataset generator is suitable for training models for `indoor_pm25`, `indoor_pm10`, and `indoor_no2`.

features:
- ventilation is applied as a multiplicative reduction to indoor-generated pollutant rates
- extractor fan/window effects are represented with interaction features
- optional indoor sources such as smoking/vaping and wood burning now affect pollutant rates
- cooking rows cannot have missing or zero emission rates
- `pm10_rate >= pm25_rate` and `indoor_pm10 >= indoor_pm25` are enforced
- indoor target concentrations are created for regression modelling


In [ ]:
# import libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# ============================================================
# Project paths
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)


In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

N_RECORDS = 50_000 # Number of synthetic records to generate
EMISSION_FACTOR_FILE = DATA_DIR / "unified_emission_factors_literature.csv"
OUTPUT_FILE = DATA_DIR / "synthetic_air_quality_dataset_final.csv"

if not EMISSION_FACTOR_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {EMISSION_FACTOR_FILE}. Put the emission-factor CSV in the data/ folder."
    )

ef = pd.read_csv(EMISSION_FACTOR_FILE)

# Standardise text fields used for joining/sample lookup
ef["cooking_method"] = ef["cooking_method"].astype(str).str.strip().str.lower()
ef["appliance_type"] = ef["appliance_type"].astype(str).str.strip().str.lower()

required_ef_cols = [
    "cooking_method", "appliance_type",
    "pm25_mg_per_min", "pm10_mg_per_min",
    "no2_mg_per_min_min", "no2_mg_per_min_max"
]
missing_cols = [c for c in required_ef_cols if c not in ef.columns]
if missing_cols:
    raise ValueError(f"Emission-factor file is missing these columns: {missing_cols}")

# Basic EF cleanup
for col in ["pm25_mg_per_min", "pm10_mg_per_min", "no2_mg_per_min_min", "no2_mg_per_min_max"]:
    ef[col] = pd.to_numeric(ef[col], errors="coerce")

ef = ef.dropna(subset=required_ef_cols).copy()
ef["pm10_mg_per_min"] = np.maximum(ef["pm10_mg_per_min"], ef["pm25_mg_per_min"] * 1.05)
ef["no2_mg_per_min_max"] = np.maximum(ef["no2_mg_per_min_max"], ef["no2_mg_per_min_min"])


In [ ]:
# -----------------------------
# 1. Base timeline
# -----------------------------
df = pd.DataFrame({
    "day": pd.date_range("2025-01-01", periods=N_RECORDS, freq="h").date,
    "start_time_local": pd.date_range("2025-01-01", periods=N_RECORDS, freq="h"),
})

df["duration_min"] = np.where(
    np.random.rand(N_RECORDS) < 0.8,
    np.random.randint(5, 61, N_RECORDS),
    np.random.randint(61, 121, N_RECORDS)
)

df["end_time_local"] = df["start_time_local"] + pd.to_timedelta(df["duration_min"], unit="m")
df["hour"] = df["start_time_local"].dt.hour
df["time_of_day"] = pd.cut(
    df["hour"],
    bins=[0, 6, 12, 18, 24],
    labels=["night", "morning", "afternoon", "evening"],
    right=False
)

def traffic_factor(hour):
    if 7 <= hour <= 9 or 17 <= hour <= 19:
        return np.random.uniform(1.3, 1.8)
    return np.random.uniform(0.7, 1.2)

def ozone_factor(hour):
    if 11 <= hour <= 16:
        return np.random.uniform(1.2, 1.6)
    return np.random.uniform(0.6, 1.0)

df["traffic_factor"] = df["hour"].apply(traffic_factor)
df["ozone_factor"] = df["hour"].apply(ozone_factor)


In [ ]:
# -----------------------------
# 2. Outdoor pollutant generation
# -----------------------------
df["outdoor_pm25"] = (
    np.random.uniform(5, 25, N_RECORDS) * df["traffic_factor"]
    + np.random.normal(0, 2, N_RECORDS)
).clip(lower=2)

pm10_ratio = np.random.uniform(1.15, 1.80, N_RECORDS)
pm10_noise = np.random.normal(0, 4, N_RECORDS)
df["outdoor_pm10"] = (df["outdoor_pm25"] * pm10_ratio + pm10_noise).clip(lower=df["outdoor_pm25"] * 1.05)

df["outdoor_no2"] = (
    np.random.uniform(10, 50, N_RECORDS) * df["traffic_factor"]
    + np.random.normal(0, 5, N_RECORDS)
).clip(lower=5)

df["outdoor_o3"] = (
    np.random.uniform(30, 80, N_RECORDS) * df["ozone_factor"]
    - 0.3 * df["outdoor_no2"]
).clip(lower=10)

df["outdoor_so2"] = np.random.uniform(2, 15, N_RECORDS)

df["outdoor_temp"] = (
    10
    + 10 * np.sin(2 * np.pi * df["hour"] / 24)
    + np.random.normal(0, 2, N_RECORDS)
)


In [ ]:
# -----------------------------------------
# 3. Indoor context and activity generation
# ------------------------------------------
def sample_activity(hour):
    # Fixed: overnight rows should not be dominated by cooking.
    if 6 <= hour <= 9:       # breakfast
        probs = [0.35, 0.50, 0.15]   # resting, cooking, cleaning
    elif 12 <= hour <= 14:   # lunch
        probs = [0.30, 0.55, 0.15]
    elif 17 <= hour <= 20:   # dinner
        probs = [0.30, 0.60, 0.10]
    elif 21 <= hour <= 23:
        probs = [0.15, 0.70, 0.15]
    else:                    # night/early morning
        probs = [0.05, 0.85, 0.10]
    return np.random.choice(["resting", "cooking", "cleaning"], p=probs)

df["motion_type"] = np.random.choice(
    ["stationary", "walking", "active_household"],
    N_RECORDS,
    p=[0.70, 0.10, 0.20]
)
df["environment_type"] = "indoor"
df["activity_type"] = df["hour"].apply(sample_activity)
df["cooking_event"] = df["activity_type"].eq("cooking")

# Defaults
df["cooking_method"] = "none"
df["appliance_type"] = "none"
df["gas_cooking_or_gas_hob"] = 0
df["cooking_duration_min"] = 0

# Use EF table as the source of truth for valid cooking combinations
valid_pairs = ef[["cooking_method", "appliance_type"]].drop_duplicates()
cook_idx = df.index[df["cooking_event"]]
sampled_pairs = valid_pairs.sample(n=len(cook_idx), replace=True, random_state=42).reset_index(drop=True)

df.loc[cook_idx, "cooking_method"] = sampled_pairs["cooking_method"].to_numpy()
df.loc[cook_idx, "appliance_type"] = sampled_pairs["appliance_type"].to_numpy()
df.loc[cook_idx, "gas_cooking_or_gas_hob"] = df.loc[cook_idx, "appliance_type"].eq("gas").astype(int).to_numpy()
df.loc[cook_idx, "cooking_duration_min"] = np.random.randint(5, 121, len(cook_idx))

# Optional non-cooking/secondary sources
df["wood_burning_event"] = np.random.choice([0, 1], N_RECORDS, p=[0.95, 0.05])
df["smoking_vaping_exposure"] = np.random.choice([0, 1], N_RECORDS, p=[0.90, 0.10])

# Ventilation variables
# Windows can be open in any activity; cooking increases the probability.
df["window_open"] = np.random.choice([0, 1], N_RECORDS, p=[0.72, 0.28])
df.loc[cook_idx, "window_open"] = np.random.choice([0, 1], len(cook_idx), p=[0.55, 0.45])

# Extractor fan is mostly cooking-related, but not perfectly correlated.
# This gives regression enough variation to learn the fan effect properly.
df["extractor_fan_on"] = 0
df.loc[cook_idx, "extractor_fan_on"] = np.random.choice([0, 1], len(cook_idx), p=[0.35, 0.65])

# Keep a very small number of non-cooking fan-on rows to avoid perfect correlation.
non_cook_idx = df.index[~df["cooking_event"]]
fan_non_cook = np.random.choice([0, 1], len(non_cook_idx), p=[0.97, 0.03])
df.loc[non_cook_idx, "extractor_fan_on"] = fan_non_cook

# Hard reset cooking-only attributes for non-cooking rows
df.loc[non_cook_idx, ["cooking_method", "appliance_type"]] = ["none", "none"]
df.loc[non_cook_idx, ["gas_cooking_or_gas_hob", "cooking_duration_min"]] = 0


In [ ]:
# -----------------------------
# 4. Raw cooking emission rates
# -----------------------------
def get_cooking_emissions(row):
    if not row["cooking_event"]:
        return 0.0, 0.0, 0.0

    subset = ef[
        (ef["cooking_method"] == row["cooking_method"])
        & (ef["appliance_type"] == row["appliance_type"])
    ]

    if subset.empty:
        return np.nan, np.nan, np.nan

    row_ef = subset.sample(1)
    pm25 = float(row_ef["pm25_mg_per_min"].iloc[0])
    pm10 = float(row_ef["pm10_mg_per_min"].iloc[0])
    no2 = np.random.uniform(
        float(row_ef["no2_mg_per_min_min"].iloc[0]),
        float(row_ef["no2_mg_per_min_max"].iloc[0])
    )
    return pm25, pm10, no2

df[["cooking_pm25_raw", "cooking_pm10_raw", "cooking_no2_raw"]] = df.apply(
    lambda row: pd.Series(get_cooking_emissions(row)), axis=1
)


In [ ]:
# ------------------------------------------
# 5. Source effects + ventilation multiplier
# ------------------------------------------
# Start with raw cooking rates
df["pm25_rate"] = df["cooking_pm25_raw"].copy()
df["pm10_rate"] = df["cooking_pm10_raw"].copy()
df["no2_rate"] = df["cooking_no2_raw"].copy()

cook_mask = df["cooking_event"]
gas_mask = cook_mask & df["appliance_type"].eq("gas")
electric_mask = cook_mask & df["appliance_type"].eq("electric")

# Cooking duration/intensity effect
duration_scale = 1 + (df["cooking_duration_min"] / 120)
df.loc[cook_mask, "pm25_rate"] *= duration_scale[cook_mask]
df.loc[cook_mask, "pm10_rate"] *= duration_scale[cook_mask]
df.loc[cook_mask, "no2_rate"] *= duration_scale[cook_mask]

# Gas cooking produces more NO2; electric cooking contributes less NO2
df.loc[gas_mask, "no2_rate"] *= np.random.uniform(1.4, 2.2, gas_mask.sum())
df.loc[electric_mask, "no2_rate"] *= np.random.uniform(0.8, 1.1, electric_mask.sum())

# Add secondary indoor sources before ventilation reduction
smoke_mask = df["smoking_vaping_exposure"].eq(1)
wood_mask = df["wood_burning_event"].eq(1)
clean_mask = df["activity_type"].eq("cleaning")

df.loc[smoke_mask, "pm25_rate"] += np.random.uniform(20, 80, smoke_mask.sum())
df.loc[smoke_mask, "pm10_rate"] += np.random.uniform(25, 95, smoke_mask.sum())
df.loc[smoke_mask, "no2_rate"] += np.random.uniform(0.5, 4.0, smoke_mask.sum())

df.loc[wood_mask, "pm25_rate"] += np.random.uniform(40, 140, wood_mask.sum())
df.loc[wood_mask, "pm10_rate"] += np.random.uniform(50, 170, wood_mask.sum())
df.loc[wood_mask, "no2_rate"] += np.random.uniform(1.0, 6.0, wood_mask.sum())

# Cleaning mostly resuspends/coarse particles, so PM10 uplift is larger than PM2.5
df.loc[clean_mask, "pm25_rate"] += np.random.uniform(0.05, 0.30, clean_mask.sum())
df.loc[clean_mask, "pm10_rate"] += np.random.uniform(0.10, 0.60, clean_mask.sum())

# Ventilation reduction is applied as a multiplier to indoor-generated pollutants.
# This fixes the issue where fan/window were correlated with high-emission events but did not visibly reduce rates.
df["vent_pm25_multiplier"] = 1.0
df["vent_pm10_multiplier"] = 1.0
df["vent_no2_multiplier"] = 1.0

fan_mask = df["extractor_fan_on"].eq(1)
window_mask = df["window_open"].eq(1)

df.loc[fan_mask, "vent_pm25_multiplier"] *= np.random.uniform(0.50, 0.80, fan_mask.sum())
df.loc[fan_mask, "vent_pm10_multiplier"] *= np.random.uniform(0.55, 0.85, fan_mask.sum())
df.loc[fan_mask, "vent_no2_multiplier"] *= np.random.uniform(0.60, 0.85, fan_mask.sum())

df.loc[window_mask, "vent_pm25_multiplier"] *= np.random.uniform(0.65, 0.90, window_mask.sum())
df.loc[window_mask, "vent_pm10_multiplier"] *= np.random.uniform(0.70, 0.92, window_mask.sum())
df.loc[window_mask, "vent_no2_multiplier"] *= np.random.uniform(0.75, 0.95, window_mask.sum())

df["pm25_rate"] *= df["vent_pm25_multiplier"]
df["pm10_rate"] *= df["vent_pm10_multiplier"]
df["no2_rate"] *= df["vent_no2_multiplier"]

# Enforce PM10 >= PM2.5 for indoor-generated rates
df["pm10_rate"] = np.maximum(
    df["pm10_rate"],
    df["pm25_rate"] * np.random.uniform(1.05, 1.30, len(df))
)


In [ ]:
# -----------------------------------------------------------------
# 6. Convert rates into indoor target concentrations for regression
# -----------------------------------------------------------------
# Outdoor infiltration: open windows increase outdoor pollutant entry, but ventilation still reduces indoor-generated sources.
df["pm25_infiltration_factor"] = np.random.uniform(0.45, 0.70, N_RECORDS)
df["pm10_infiltration_factor"] = np.random.uniform(0.35, 0.60, N_RECORDS)
df["no2_infiltration_factor"] = np.random.uniform(0.40, 0.70, N_RECORDS)

df.loc[window_mask, "pm25_infiltration_factor"] += np.random.uniform(0.05, 0.20, window_mask.sum())
df.loc[window_mask, "pm10_infiltration_factor"] += np.random.uniform(0.05, 0.18, window_mask.sum())
df.loc[window_mask, "no2_infiltration_factor"] += np.random.uniform(0.05, 0.20, window_mask.sum())

for col in ["pm25_infiltration_factor", "pm10_infiltration_factor", "no2_infiltration_factor"]:
    df[col] = df[col].clip(0.05, 0.95)

# Convert source emission rate to concentration contribution.
# This keeps the synthetic targets in a plausible modelling range 
pm_source_scale = df["duration_min"] / 60
no2_source_scale = df["duration_min"] / 60

df["indoor_pm25"] = (
    df["outdoor_pm25"] * df["pm25_infiltration_factor"]
    + df["pm25_rate"] * pm_source_scale
    + np.random.normal(0, 1.5, N_RECORDS)
).clip(lower=0)

df["indoor_pm10"] = (
    df["outdoor_pm10"] * df["pm10_infiltration_factor"]
    + df["pm10_rate"] * pm_source_scale
    + np.random.normal(0, 2.0, N_RECORDS)
).clip(lower=0)

df["indoor_no2"] = (
    df["outdoor_no2"] * df["no2_infiltration_factor"]
    + df["no2_rate"] * no2_source_scale
    + np.random.normal(0, 1.2, N_RECORDS)
).clip(lower=0)

# Enforce PM10 >= PM2.5 for final indoor concentrations
df["indoor_pm10"] = np.maximum(
    df["indoor_pm10"],
    df["indoor_pm25"] * np.random.uniform(1.02, 1.20, N_RECORDS)
)

# Interaction features for regression
df["cooking_x_window_open"] = df["cooking_event"].astype(int) * df["window_open"].astype(int)
df["cooking_x_extractor_fan"] = df["cooking_event"].astype(int) * df["extractor_fan_on"].astype(int)
df["gas_cooking_x_extractor_fan"] = df["gas_cooking_or_gas_hob"].astype(int) * df["extractor_fan_on"].astype(int)


In [ ]:
# -----------------------------
# 7. Integrity and validation checks
# -----------------------------
cooking_required = [
    "cooking_method", "appliance_type", "cooking_duration_min",
    "pm25_rate", "pm10_rate", "no2_rate",
    "indoor_pm25", "indoor_pm10", "indoor_no2"
]

bad_cooking_nulls = df.loc[df["cooking_event"], cooking_required].isna().any(axis=1)
assert not bad_cooking_nulls.any(), "Some cooking rows have missing fields or emission/target values."

bad_cooking_zero = df[
    df["cooking_event"]
    & (
        (df["pm25_rate"] <= 0)
        | (df["pm10_rate"] <= 0)
        | (df["no2_rate"] <= 0)
    )
]
assert len(bad_cooking_zero) == 0, "Some cooking rows have zero or negative emission rates."

bad_non_cook = df[
    (~df["cooking_event"])
    & (
        (df["cooking_method"] != "none")
        | (df["appliance_type"] != "none")
        | (df["gas_cooking_or_gas_hob"] != 0)
        | (df["cooking_duration_min"] != 0)
    )
]
assert len(bad_non_cook) == 0, "Some non-cooking rows still carry cooking-only attributes."

assert (df["pm10_rate"] >= df["pm25_rate"]).all(), "PM10 rate is lower than PM2.5 rate in some rows."
assert (df["indoor_pm10"] >= df["indoor_pm25"]).all(), "Indoor PM10 is lower than indoor PM2.5 in some rows."

# Similar-activity validation: check ventilation on indoor-generated rates within cooking groups.
validation_cols = ["activity_type", "cooking_method", "appliance_type"]
ventilation_summary = (
    df[df["cooking_event"]]
    .groupby(validation_cols + ["extractor_fan_on", "window_open"], observed=True)
    [["pm25_rate", "pm10_rate", "no2_rate", "indoor_pm25", "indoor_pm10", "indoor_no2"]]
    .median()
    .reset_index()
)

print("Integrity checks passed.")
print("Rows:", len(df))
print("Cooking rows:", int(df["cooking_event"].sum()))
print("PM10 >= PM2.5 rate:", bool((df["pm10_rate"] >= df["pm25_rate"]).all()))
print("Indoor PM10 >= Indoor PM2.5:", bool((df["indoor_pm10"] >= df["indoor_pm25"]).all()))

ventilation_summary.head(10)


In [ ]:
# ------------------------------------
# 8. Final column cleanup and export
# ------------------------------------
# Keep helper columns that are useful for checking/model explanation, but remove raw temporary generators.
df = df.drop(columns=["traffic_factor", "ozone_factor", "hour"])

# Optional: put target variables near the end for readability
cols = [c for c in df.columns if c not in ["indoor_pm25", "indoor_pm10", "indoor_no2"]]
df = df[cols + ["indoor_pm25", "indoor_pm10", "indoor_no2"]]

df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved dataset to: {OUTPUT_FILE}")
df.head()
